In [1]:
from csv import DictReader

with open('source_data.csv', 'r') as csv_file:
    reader = DictReader(csv_file)
    blend_components = [r for r in reader]
csv_file.close()

for b in blend_components:
    b['Volume%'] = float(b['Volume%']) * 0.01
_volsum = sum([b['Volume%'] for b in blend_components])
for b in blend_components:
    b['Volume%'] = b['Volume%'] / _volsum

In [2]:
with open('properties_master.csv', 'r') as csv_file:
    reader = DictReader(csv_file)
    all_mols = [r for r in reader]
csv_file.close()

for compound in blend_components:
    compound['Exp. CN'] = '-'
    compound['Exp. YSI'] = '-'
    compound['Exp. LHV (MJ/kg)'] = '-'
    compound['Exp. KV (cSt)'] = '-'
    compound['Exp. CP (deg. C)'] = '-'
    compound['Exp. FP (deg. C)'] = '-'
    compound['Exp. BP (deg. C)'] = '-'
    for mol in all_mols:
        if compound['CAS Number'] == mol['cas']:
            compound['Exp. CN'] = mol['properties.cetane_number.value']
            compound['Exp. YSI'] = mol['properties.ysi_unified.value']
            compound['Exp. LHV (MJ/kg)'] = '-'  # LHV data not in master DB (yet)
            compound['Exp. KV (cSt)'] = mol['properties.kinematic_viscosity.value']
            compound['Exp. CP (deg. C)'] = '-'  # CP data not in master DB (yet)
            compound['Exp. FP (deg. C)'] = mol['properties.flash_point.value']
            compound['Exp. BP (deg. C)'] = mol['properties.boiling_point.value']
            break

In [3]:
from ecnet.datasets import QSPRDataset

smiles = [b['SMILES'] for b in blend_components]
ds = QSPRDataset(smiles, [0.0 for _ in range(len(smiles))], backend='alvadesc')

In [4]:
from copy import deepcopy
from ecnet.model import load_model

with open('_models/cn_desc.txt', 'r') as desc_file:
    desc_idx = desc_file.readlines()
desc_file.close()
desc_idx = [int(d.replace('\n', '')) for d in desc_idx]
ds_eval = deepcopy(ds)
ds_eval.set_desc_index(desc_idx)
ecnet = load_model('_models/cn.pt')
cn_preds = ecnet.forward(ds_eval.desc_vals).detach().numpy()
cn_preds = [c[0] for c in cn_preds]

with open('_models/ysi_desc.txt', 'r') as desc_file:
    desc_idx = desc_file.readlines()
desc_file.close()
desc_idx = [int(d.replace('\n', '')) for d in desc_idx]
ds_eval = deepcopy(ds)
ds_eval.set_desc_index(desc_idx)
ecnet = load_model('_models/ysi.pt')
ysi_preds = ecnet.forward(ds_eval.desc_vals).detach().numpy()
ysi_preds = [c[0] for c in ysi_preds]

with open('_models/lhv_desc.txt', 'r') as desc_file:
    desc_idx = desc_file.readlines()
desc_file.close()
desc_idx = [int(d.replace('\n', '')) for d in desc_idx]
ds_eval = deepcopy(ds)
ds_eval.set_desc_index(desc_idx)
ecnet = load_model('_models/lhv.pt')
lhv_preds = ecnet.forward(ds_eval.desc_vals).detach().numpy()
lhv_preds = [c[0] for c in lhv_preds]

with open('_models/kv_desc.txt', 'r') as desc_file:
    desc_idx = desc_file.readlines()
desc_file.close()
desc_idx = [int(d.replace('\n', '')) for d in desc_idx]
ds_eval = deepcopy(ds)
ds_eval.set_desc_index(desc_idx)
ecnet = load_model('_models/kv.pt')
kv_preds = ecnet.forward(ds_eval.desc_vals).detach().numpy()
kv_preds = [c[0] for c in kv_preds]

with open('_models/cp_desc.txt', 'r') as desc_file:
    desc_idx = desc_file.readlines()
desc_file.close()
desc_idx = [int(d.replace('\n', '')) for d in desc_idx]
ds_eval = deepcopy(ds)
ds_eval.set_desc_index(desc_idx)
ecnet = load_model('_models/cp.pt')
cp_preds = ecnet.forward(ds_eval.desc_vals).detach().numpy()
cp_preds = [c[0] for c in cp_preds]

with open('_models/fp_desc.txt', 'r') as desc_file:
    desc_idx = desc_file.readlines()
desc_file.close()
desc_idx = [int(d.replace('\n', '')) for d in desc_idx]
ds_eval = deepcopy(ds)
ds_eval.set_desc_index(desc_idx)
ecnet = load_model('_models/fp.pt')
fp_preds = ecnet.forward(ds_eval.desc_vals).detach().numpy()
fp_preds = [c[0] for c in fp_preds]

with open('_models/bp_desc.txt', 'r') as desc_file:
    desc_idx = desc_file.readlines()
desc_file.close()
desc_idx = [int(d.replace('\n', '')) for d in desc_idx]
ds_eval = deepcopy(ds)
ds_eval.set_desc_index(desc_idx)
ecnet = load_model('_models/bp.pt')
bp_preds = ecnet.forward(ds_eval.desc_vals).detach().numpy()
bp_preds = [c[0] for c in bp_preds]

for idx, compound in enumerate(blend_components):
    compound['Pred. CN'] = cn_preds[idx]
    compound['Pred. YSI'] = ysi_preds[idx]
    compound['Pred. LHV (MJ/kg)'] = lhv_preds[idx]
    compound['Pred. KV (cSt)'] = kv_preds[idx]
    compound['Pred. CP (deg. C)'] = cp_preds[idx]
    compound['Pred. FP (deg. C)'] = fp_preds[idx]
    compound['Pred. BP (deg. C)'] = bp_preds[idx]
    if compound['Exp. CN'] != '-':
        compound['Abs. Err. CN'] = abs(float(compound['Exp. CN']) - float(compound['Pred. CN']))
    else:
        compound['Abs. Err. CN'] = '-'
    if compound['Exp. YSI'] != '-':
        compound['Abs. Err. YSI'] = abs(float(compound['Exp. YSI']) - float(compound['Pred. YSI']))
    else:
        compound['Abs. Err. YSI'] = '-'
    if compound['Exp. LHV (MJ/kg)'] != '-':
        compound['Abs. Err. LHV (MJ/kg)'] = abs(float(compound['Exp. LHV (MJ/kg)']) - float(compound['Pred. LHV (MJ/kg)']))
    else:
        compound['Abs. Err. LHV (MJ/kg)'] = '-'
    if compound['Exp. KV (cSt)'] != '-':
        compound['Abs. Err. KV (cSt)'] = abs(float(compound['Exp. KV (cSt)']) - float(compound['Pred. KV (cSt)']))
    else:
        compound['Abs. Err. KV (cSt)'] = '-'
    if compound['Exp. CP (deg. C)'] != '-':
        compound['Abs. Err. CP (deg. C)'] = abs(float(compound['Exp. CP (deg. C)']) - float(compound['Pred. CP (deg. C)']))
    else:
        compound['Abs. Err. CP (deg. C)'] = '-'
    if compound['Exp. FP (deg. C)'] != '-':
        compound['Abs. Err. FP (deg. C)'] = abs(float(compound['Exp. FP (deg. C)']) - float(compound['Pred. FP (deg. C)']))
    else:
        compound['Abs. Err. FP (deg. C)'] = '-'
    if compound['Exp. BP (deg. C)'] != '-':
        compound['Abs. Err. BP (deg. C)'] = abs(float(compound['Exp. BP (deg. C)']) - float(compound['Pred. BP (deg. C)']))
    else:
        compound['Abs. Err. BP (deg. C)'] = '-'

In [5]:
from ecnet.blends import cetane_number, yield_sooting_index, lower_heating_value, kinematic_viscosity, cloud_point
from ecnet.blends.equations import linear_blend_ave

vol_perc = [b['Volume%'] for b in blend_components]

cn_blend = cetane_number(cn_preds, vol_perc)
ysi_blend = yield_sooting_index(ysi_preds, vol_perc)
lhv_blend = lower_heating_value(lhv_preds, vol_perc)
kv_blend = kinematic_viscosity(kv_preds, vol_perc)
cp_blend = cloud_point(cp_preds, vol_perc)

# FP blend assumed linear per https://www.sciencedirect.com/science/article/abs/pii/S0016236107004966
fp_blend = linear_blend_ave(fp_preds, vol_perc)

# Assuming linear BP blend (for now, until we come up with a model/equation)
bp_blend = linear_blend_ave(bp_preds, vol_perc)

blend_components.append(deepcopy(blend_components[0]))
for key in list(blend_components[0].keys()):
    blend_components[-1][key] = '-'
blend_components[-1]['SMILES'] = 'BLEND'
blend_components[-1]['Pred. CN'] = cn_blend
blend_components[-1]['Pred. YSI'] = ysi_blend
blend_components[-1]['Pred. LHV (MJ/kg)'] = lhv_blend
blend_components[-1]['Pred. KV (cSt)'] = kv_blend
blend_components[-1]['Pred. CP (deg. C)'] = cp_blend
blend_components[-1]['Pred. FP (deg. C)'] = fp_blend
blend_components[-1]['Pred. BP (deg. C)'] = bp_blend

In [6]:
from csv import DictWriter

headers = list(blend_components[0].keys())
with open('results.csv', 'w') as csv_file:
    writer = DictWriter(csv_file, headers, delimiter=',', lineterminator='\n')
    writer.writeheader()
    writer.writerows(blend_components)
csv_file.close()